# Torchinductor Walkthrough 

We are going to walk through the various stages of torch.compile IR. 

## Pre-grad: 

Dynamo, as it traces the graph, builds up an IR representation that is essentially equivalent to what exists in user code. At
this point:
- IR has not been normalized: both `torch.nn.functional.relu()` and `x.relu()` may both exist 
- Mutation and unclear aliasing relationships still remain. For instance - operators like `.to` may return the input or clone it.
- Decompositions have not yet been applied. Pytorch's [2000](https://dev-discuss.pytorch.org/t/where-do-the-2000-pytorch-operators-come-from-more-than-you-wanted-to-know/373) operators still exist, as opposed to the much more limited set operators post decomposition.
- the forward and backward have not been traced out. It's unclear which operators will require what saved activations.
- Tensor Subclasses may exist in the graph with their semantics not yet traced through
- Similarly, Automated Mixed Precision, or other global modes which affect dispatching, will not yet be present.

For all of the above reasons, we discourage graph manipulation on this stage of the ir. 

In [1]:
import os
os.environ['TORCH_LOGS_FORMAT'] = "[%(levelname)s]:%(message)s"
import torch
torch._inductor.config.force_disable_caches = True

torch._logging.set_logs(pre_grad_graphs=True)


@torch.compile()
def matmul(x):
    out = torch.nn.functional.linear(x, x)
    return out.relu(), torch.nn.functional.relu(out.add_(1)).to(torch.float16)


inp = torch.rand([256, 256], device="cuda", requires_grad=True)
out = matmul(inp)

torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


[DEBUG]:TRACED GRAPH
 ===== BEFORE PRE GRAD =====
 torch/fx/_lazy_graph_module.py class GraphModule(torch.nn.Module):
    def forward(self, L_x_: "f32[256, 256][256, 1]cuda:0"):
        l_x_ = L_x_

        # File: <cell>:11 in matmul, code: out = torch.nn.functional.linear(x, x)
        out: "f32[256, 256][256, 1]cuda:0" = torch._C._nn.linear(l_x_, l_x_);  l_x_ = None

        # File: <cell>:12 in matmul, code: return out.relu(), torch.nn.functional.relu(out.add_(1)).to(torch.float16)
        relu: "f32[256, 256][256, 1]cuda:0" = out.relu()
        add_: "f32[256, 256][256, 1]cuda:0" = out.add_(1);  out = None
        relu_1: "f32[256, 256][256, 1]cuda:0" = torch.nn.functional.relu(add_);  add_ = None
        to: "f16[256, 256][256, 1]cuda:0" = relu_1.to(torch.float16);  relu_1 = None
        return (relu, to)




## Joint Graph 

To convert from the pre grad to joint graph representation, we trace through the graph with a custom `TorchDispatchMode` that builds up a graph on the fly. See [ProxyTorchDispatchMode](https://github.com/pytorch/pytorch/blob/6f6acb412828844ee3bcdbf277283144faba2524/torch/fx/experimental/proxy_tensor.py#L1379-L1410). During this tracing, we also functionalize the graph. For a more detailed overview of Tensor Subclasses, check out this existing [colab presentation](https://colab.research.google.com/drive/1zjAisRrc8R6uixKsrs1DRm3lwz5MWN68).


The Joint Graph is functional and made up of aten-IR. `torch.nn.functional.relu` is transformed to `aten.relu`, and the specific overload `aten.relu.default` corresponding to a particular input schemas has been determined. [CompositeImplicit Operators](https://github.com/pytorch/pytorch/blob/main/aten/src/ATen/native/README.md#dispatch), which internally dispatch to operators (`.to` for instance), will have been removed from the graph. Decompositions have also been applied. 

The Joint Graph is the first stage of the IR that inductor applies transformations to. 

**When to use joint graph passes:**

* **if you need to change both the forward and backward implementation of an operator**
* **if you expect your pass to decisions the partitioner makes about what tensors are saved, and what is recomputed**

Joint Graph passes should keep the graph functional, and composed of aten-ir. 

[Inductor Joint Graph Passes](https://github.com/pytorch/pytorch/blob/41de0f2eaf97fa9419190b93dc0b5b3b051a9adf/torch/_inductor/fx_passes/joint_graph.py).




In [2]:
torch._dynamo.reset()
torch._logging.set_logs(pre_grad_graphs=False, aot_joint_graph=True)

out = matmul(inp)

torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(
[INFO]:TRACED GRAPH
 ===== Joint graph 1 =====
 torch/fx/_lazy_graph_module.py class inner_f(torch.nn.Module):
    def forward(self, primals, tangents):
        primals_1: "f32[256, 256][256, 1]cuda:0"; tangents_1: "f32[256, 256][256, 1]cuda:0"; tangents_2: "f16[256, 256][256, 1]cuda:0";

        primals_1, tangents_1, tangents_2, = fx_pytree.tree_flatten_spec([primals, tangents], self._in_spec)
        # File: <cell>:11 in matmul, code: out = torch.nn.functional.linear(x, x)
        permute: "f32[256, 256][1, 256]cuda:0" = torch.ops.aten.permute.default(primals_1, [1, 0])
        mm: "f32[256, 256][256, 1]cuda:0" = torch.ops.aten.mm.default(primals_1, permute)

        # File: <cell>:12 in matmul, code: return out.relu(), torch.nn.functional.relu(out.add_(1)).to(torch.float16)
        relu: "f32[256, 256][256, 1]cuda:0" = torch.ops.aten.relu.default(mm)
        a

# (WIP) Post Grad Passes

After the partitioner has partitioned the joint graph into a separate forward and backward graph, we apply our post grad passes. 

There are two stages of post grad passes: functional and non-functional. 






# TorchInductor Scheduler

To start, lets look at a very simple example, a decomposed gelu. 


In [3]:
torch._logging.set_logs(ir_pre_fusion=True, ir_post_fusion=True)


@torch.compile()
def foo(x):
    mul = torch.ops.aten.mul.Tensor(x, 0.5)
    mul_1 = torch.ops.aten.mul.Tensor(x, 0.7071067811865476)
    erf = torch.ops.aten.erf.default(mul_1)
    add = torch.ops.aten.add.Tensor(erf, 1)
    mul_2 = torch.ops.aten.mul.Tensor(mul, add)
    return (mul_2,)


_ = foo(torch.rand([32, 32], device="cuda"))

[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', 32*d0 + d1, {d0: 32, d1: 32})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', 32*d0 + d1, {d0: 32, d1: 32})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (1024, 1)
op0.sizes = ([32, 32], [])
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
class op0_loop_body:
    var_ranges = {p0: 32, p1: 32}
    index0 = 32*p0 + p1
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg0_1', get_index)
        constant = ops.constant(0.5, torch.float32)
        mul = ops.mul(l

[INFO]:AFTER FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', c0, {c0: 1024})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', c0, {c0: 1024})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (1024, 1)
op0.sizes = ((1024,), ())
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[32, 32], stride=[32, 1])
class op0_loop_body:
    var_ranges = {p0: 1024}
    index0 = p0
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg0_1', get_index)
        constant = ops.constant(0.5, torch.float32)
        mul = ops.mul(load, constant)
        get_index_1 = self.ge

Oops, no fusion occurred - there was only a single Scheduler Node before and after fusion.

That is because during aten IR lowering to Inductor IR, we do some amount of inlining as we lower. We only inline pointwise objects. 

Recomputing operations can allow us to improve fusion. Here, on the left, we assume that `clone` cannot be fused. By inlining `cos` and `relu` into their use in `add`, we avoid having to materialize them in global memory. 

This can be especially important when broadcasts result in a output of pointwise node being much larger than its inputs, and in the backward, where we often save smaller tensors that get reused multiple times, and broadcasted/converted to a larger dtype.

<!-- TODO: inlining diagram (clone not fusible; cos and relu inlined into add) still needs to be added. -->






In some cases, inlining also avoids current scheduler-level limitations where we wouldn't fuse two nodes. 


To update the inlining heuristcs, see `config.realize_reads_threshold`, `config.realize_opcount_threshold`, and `realize_acc_reads_threshold` [here](https://github.com/pytorch/pytorch/blob/f0c9b3385d99a74075dae3cc3e53d79b619bee7f/torch/_inductor/config.py#L505-L512).


## Scheduler

The scheduler has 2 components. There’s the back-end agnostic component (Scheduler in scheduler.py), and the back-end specific component (TritonScheduling in codegen/triton.py). Different backends may have different heuristics on what is fusable and how profitable the fusions might be. 


The first thing the scheduler does is convert lowering IR `ComputedBuffer` nodes into schedule.SchedulerNode nodes with corresponding `SchedulerBuffer`. (this happens [here](https://github.com/pytorch/pytorch/blob/main/torch/_inductor/scheduler.py#L1025-L1068))

There are a few important fields on the SchedulerNode, in addition to various use/def metadata you might be familiar.
```
class SchedulerNode:
    _sizes: tuple[Sequence[sympy.Expr], ...]
    _body: LoopBody
    read_writes: dependencies.ReadWrites
```

`_sizes` correspond to the iteration variables inductor will codegen. The first sequence is the pointwise iteration sizes, and the second sequence is the reduction iteration size. 

`_body` is the persistent representation of Inductor IR that we will do loop level modifications to. For instance, [applying a new loop order](https://github.com/pytorch/pytorch/blob/8cb6957e019086c1f6e39941047ead6e29f6f402/torch/_inductor/scheduler.py#L1108-L1113).

`_read_writes`: are the global reads and writes of the ir. 

In order to fuse we need to have compatible iteration variables, and we need a matching read and write which indicates fusing the two nodes would result in reduced global memory access. 

Lets take a look at an example.


In [4]:
torch._logging.set_logs(ir_pre_fusion=True, fusion=True, ir_post_fusion=True)

@torch.compile(dynamic=True)
def foo(x, y):
    return x.sum(dim=1) + (y + 3).sum(dim=1)

_ = foo(torch.rand([10, 50], device="cuda"), torch.rand([10, 60], device="cuda"))


[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', d0, {d0: s77})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg2_1', d0*s27 + d1, {d0: s77, d1: s27})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[s77], stride=[1])
    buf0.users = [NodeUser(node=SchedulerNode(name='op2'), can_inplace=True, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (s77, s27)
op0.sizes = ([s77], [s27])
arg2_1_layout = FixedLayout('cuda:0', torch.float32, size=[s77, s27], stride=[s27, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[s77], stride=[1])
class op0_loop_body:
    var_ranges = {p0: s77, p1: s27}
    index0 = p0*s27 + p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg2_1', get_index)
        reduction = ops.reduction(torch.float32, torch.float

[DEBUG]:===== attempting fusion (1/10): 3 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[s27]', 'sum', 'origins=OrderedSet([sum_1])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '           ^^^^^^^^^^^^', '', '}'])


[DEBUG]:  SchedulerNode(name='op1'), Reduction(['[s94]', 'sum', 'origins=OrderedSet([sum_2, add])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '                          ^^^^^^^^^^^^^^^^^^', '', '}', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '                           ~~^~~', '', '}'])


[DEBUG]:  SchedulerNode(name='op2'), Pointwise(['[s77]', 'origins=OrderedSet([add_1])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '           ~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~', '', '}'])


... (14 outputs left out)

[DEBUG]:  FusedSchedulerNode(nodes=op1_op2), snodes: ['SchedulerNode(name=\'op1\'), Reduction([\'[s94]\', \'sum\', \'origins=OrderedSet([sum_2, add])\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'                          ^^^^^^^^^^^^^^^^^^\', \'\', \'}\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'                           ~~^~~\', \'\', \'}\'])', 'SchedulerNode(name=\'op2\'), Pointwise([\'[s77]\', \'origins=OrderedSet([add_1])\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'           ~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~\', \'\', \'}\'])']


[DEBUG]:cannot fuse op0 with op1_op2: numel/rnumel mismatch (reduce) (s77, s77), (s27, s94)


[DEBUG]:found 0 possible fusions


[INFO]:AFTER FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', c0, {c0: s77})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg2_1', c0, {c0: s27*s77})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[s77], stride=[1])
    buf0.users = [NodeUser(node=SchedulerNode(name='op2'), can_inplace=True, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (s77, s27)
op0.sizes = ((s77,), (s27,))
arg2_1_layout = FixedLayout('cuda:0', torch.float32, size=[s77, s27], stride=[s27, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[s77], stride=[1])
class op0_loop_body:
    var_ranges = {p0: s77, p1: s27}
    index0 = p0*s27 + p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg2_1', get_index)
        reduction = ops.reduction(torch.float32, torch.float32, 'sum', lo


There are three nodes before fusion, corresponding to the two sums, and the addition of the sum. Note, that the `y + 3` was inlined into the reduction. 

We were able to fuse one of the reductions with the add, then, we are unable to fuse the subsequent reduction. 

See: `[DEBUG]:cannot fuse op0 with op1_op2: numel/rnumel mismatch (reduce) (s77, s77), (s27, s94)`

The reduction iteration variables between the first sum and second sum differ. [Here](https://github.com/pytorch/pytorch/blob/8cb6957e019086c1f6e39941047ead6e29f6f402/torch/_inductor/codegen/simd.py#L1091-L1225) is the code that checks this.

If we re-run this example with the same sizes we should see fusion.

In [5]:
torch._logging.set_logs(ir_pre_fusion=False, fusion=True, ir_post_fusion=False)

@torch.compile(dynamic=True)
def foo(x, y):
    return x.sum(dim=1) + (y + 3).sum(dim=1)

_ = foo(torch.rand([10, 60], device="cuda"), torch.rand([10, 60], device="cuda"))


[DEBUG]:===== attempting fusion (1/10): 3 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[s27]', 'sum', 'origins=OrderedSet([sum_1])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '           ^^^^^^^^^^^^', '', '}'])


[DEBUG]:  SchedulerNode(name='op1'), Reduction(['[s27]', 'sum', 'origins=OrderedSet([sum_2, add])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '                          ^^^^^^^^^^^^^^^^^^', '', '}', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '                           ~~^~~', '', '}'])


[DEBUG]:  SchedulerNode(name='op2'), Pointwise(['[s77]', 'origins=OrderedSet([add_1])', 'stack_traces = {', '  File "<cell>", line 5, in foo', '    return x.sum(dim=1) + (y + 3).sum(dim=1)', '           ~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~', '', '}'])


[DEBUG]:found 2 possible fusions


... (3 outputs left out)

[DEBUG]:===== fusion complete (1 iterations) =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  FusedSchedulerNode(nodes=op0_op1_op2), snodes: ['SchedulerNode(name=\'op0\'), Reduction([\'[s27]\', \'sum\', \'origins=OrderedSet([sum_1])\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'           ^^^^^^^^^^^^\', \'\', \'}\'])', 'SchedulerNode(name=\'op1\'), Reduction([\'[s27]\', \'sum\', \'origins=OrderedSet([sum_2, add])\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'                          ^^^^^^^^^^^^^^^^^^\', \'\', \'}\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'                           ~~^~~\', \'\', \'}\'])', 'SchedulerNode(name=\'op2\'), Pointwise([\'[s77]\', \'origins=OrderedSet([add_1])\', \'stack_traces = {\', \'  File "<cell>", line 5, in foo\', \'    return x.sum(dim=1) + (y + 3).sum(dim=1)\', \'           ~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~\', \'\', \'}\'])']


[DEBUG]:found 0 possible fusions


Similarly, if the, reads and writes were accessed differently, we might not fuse. We do have an option to reorder loops in the scheduler to make fusion occur, which we are working on enabling by defualt. `loop_ordering_after_fusion`.

Now that we have the basics of SchedulerNode down, lets take a higher level look at how the Scheduler works. At a high level, the Scheduler has a single objective: minimize global reads and writes. 

There are essentially three parts:

-  [Generate possible fusions](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L3164-L3212) by grouping nodes which access the same buffer

```
buffer_names_grouping = collections.defaultdict(list)
for node in nodes:
    if self.unfusable_node(node):
        continue
    for buf in node.used_buffer_names():
        buffer_names_grouping[buf].append(node)
for node_grouping in buffer_names_grouping.values():
    check_all_pairs(node_grouping)
```


- [Rank possible fusions](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/choices.py#L378-L413) by amount of global memory saved, 


```
Assign a score (higher comes first) to the fusion of node1 and node2.
When different fusions conflict with each other, this is the way we
decide what order to run them in.

Our current score is based on:
- The type of fusion (template/reduction/etc)
- Estimate of the saved memory operations
- Fusions closer together in original graph order
```


- [Try to fuse nodes in order of the above ranking](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L3070-L3080)

```
for node1, node2 in self.get_possible_fusions(nodes):
    resolve_pending_fusions(node1, node2)
    node1 = self.get_fused_node(node1)
    node2 = self.get_fused_node(node2)

    if self.can_fuse(node1, node2) and not self.will_fusion_create_cycle(
        node1, node2
    ):
```

When we successfully fuse two SchedulerNodes, we create a [FusedSchedulerNode](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L1322) :


```
class FusedSchedulerNode(BaseSchedulerNode):
    """
    This is a "fake" scheduler node that represents a group of scheduler nodes
    that are meant to be fused together. The way it does this is by maintaining
    its unmet dependencies as the union of its constituent nodes.
    """

    snodes: list[BaseSchedulerNode]
```



# Case Study: epilogue aware matmul choice

In autotuning between Cublas and Triton Templates for matmuls, we take into account the surrounding possible fusions before deciding which kernel to choose. 

During scheduling time we invoke [speedup_by_fusion](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L2750-L2756) which for matmuls will benchmark the possible fusion and finalize the choice as a TritonTemplate if a fusion is profitable. 

Below, a triton template for a matmul is not profitable on its own, but becomes profitable with fusion. 


In [6]:
import torch
torch._logging.set_logs(fusion=True)


@torch.compile(mode='max-autotune-no-cudagraphs')
def foo(x, y):
    return (x @ y).relu()


x = torch.rand([8192, 1024], device="cuda", dtype=torch.float16)
y = torch.rand([1024, 1024], device="cuda", dtype=torch.float16)

_ = foo(x, y)

[DEBUG]:===== attempting fusion (1/10): 2 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0')


[DEBUG]:  SchedulerNode(name='op1'), Pointwise(['[8192, 1024]', 'origins=OrderedSet([relu])', 'stack_traces = {', '  File "<cell>", line 7, in foo', '    return (x @ y).relu()', '           ^^^^^^^^^^^^^^', '', '}'])


[DEBUG]:found 1 possible fusions


Autotune Choices Stats:
{"num_choices": 22, "num_triton_choices": 21, "best_kernel": "mm", "best_time": 0.029952000826597214, "best_triton_pos": 1, "best_triton_time": 0.03372799977660179, "best_triton_kernel": "triton_mm_17", "best_triton_kernel_desc": "ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BLOCK_M=128, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4"}
AUTOTUNE mm(8192x1024, 1024x1024)
strides: [1024, 1], [1024, 1]
dtypes: torch.float16, torch.float16
  mm 0.0300 ms 100.0% 
  triton_mm_17 0.0337 ms 88.8% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BLOCK_M=128, BLOCK_N=128, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=3, num_warps=4
  triton_mm_20 0.0338 ms 88.6% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BLOCK_M=128, BLOCK_N=256, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=4, num_warps=8
  triton_mm_18 0.0387 ms 77.4% ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=64, BLOCK_M=128, BLOCK_N

... (3 outputs left out)

[DEBUG]:===== fusion complete (1 iterations) =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  FusedSchedulerNode(nodes=op0_op1), snodes: ["SchedulerNode(name='op0')", 'SchedulerNode(name=\'op1\'), Pointwise([\'[8192, 1024]\', \'origins=OrderedSet([relu])\', \'stack_traces = {\', \'  File "<cell>", line 7, in foo\', \'    return (x @ y).relu()\', \'           ^^^^^^^^^^^^^^\', \'\', \'}\'])']


[DEBUG]:found 0 possible fusions


Because compilation of Triton Matmuls is expensive, there are actually three possible returns from `speedup_by_fusion`:

- False
- True
- Callable that will return True or False

We async compile any epilogue fusion, and store it as [a pending fusion](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L3022-L3027). Then, before we fuse either of its two nodes, or at the ending of the fusion, we resolve all pending fusions

Any matmuls that are unfused, either because lack of fusion or lack of profitability, will be finalized post fusion and pre-codegen in [finalize_multi_template_buffers](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L2086). 


# Combo Kernels / Horizontal Fusion

The scheduler fusion examples above are *vertical* fusion: two nodes share a producer/consumer relationship or a common buffer access pattern, so fusing them can remove a global read/write.

Combo kernels are a horizontal fusion pass. They target independent scheduler nodes that do not share tensors and do not have data dependencies. There is no loop-body rewrite to make them read the same data; the win is that several small kernels can share one launch.

The relevant knobs are:

- `torch._inductor.config.combo_kernels`: enable combo kernel generation.
- `torch._inductor.config.benchmark_combo_kernel`: optionally require a benchmarked speedup before accepting a combo.
- `torch._inductor.config.combo_kernel_per_subkernel_blocks`: let each subkernel keep its own autotuned block sizes.
- `test/inductor/test_combo_kernels.py`: the main test coverage for the feature.


In [7]:
import os

os.environ.setdefault("TORCH_LOGS_FORMAT", "[%(levelname)s]:%(message)s")

import torch
from torch._inductor import config as inductor_config

inductor_config.force_disable_caches = True


def set_demo_logs():
    torch._logging.set_logs(
        fusion=True,
        ir_pre_fusion=True,
        ir_post_fusion=True,
        output_code=True,
        schedule=True,
    )


def run_combo_kernel_demo(combo: bool):
    torch._dynamo.reset()
    set_demo_logs()

    @torch.compile(fullgraph=True)
    def independent_reductions(a, b):
        return a.sum(dim=-1), b.sum(dim=-1)

    # Two independent reductions with different reduction sizes.
    a = torch.randn(1024, 64, device="cuda")
    b = torch.randn(1024, 512, device="cuda")

    with inductor_config.patch({
        "combo_kernels": combo,
        "benchmark_combo_kernel": False,
        "combo_kernel_per_subkernel_blocks": combo,
        "triton.unique_kernel_names": True,
    }):
        out = independent_reductions(a, b)
        torch.cuda.synchronize()
        return out


for combo in (False, True):
    print(f"\n=== combo_kernels={combo} ===")
    _ = run_combo_kernel_demo(combo)



=== combo_kernels=False ===


torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(
[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', d0, {d0: 1024})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', 64*d0 + d1, {d0: 1024, d1: 64})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[1024], stride=[1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (1024, 64)
op0.sizes = ([1024], [64])
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[1024, 64], stride=[64, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[1024], stride=[1])
class op0_loop_body:
    var_ranges = {p0: 1024, p1: 64}
    index0 = 64*p0 + p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index('index0')

[DEBUG]:===== attempting fusion (1/10): 2 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[64]', 'sum', 'origins=OrderedSet([sum_1])', 'stack_traces = {', '  File "<cell>", line 27, in independent_reductions', '    return a.sum(dim=-1), b.sum(dim=-1)', '           ^^^^^^^^^^^^^', '', '}'])


[DEBUG]:  SchedulerNode(name='op1'), Reduction(['[512]', 'sum', 'origins=OrderedSet([sum_2])', 'stack_traces = {', '  File "<cell>", line 27, in independent_reductions', '    return a.sum(dim=-1), b.sum(dim=-1)', '                          ^^^^^^^^^^^^^', '', '}'])


... (26 outputs left out)

[INFO]:AFTER FUSION
op0_op1: ForeachKernelSchedulerNode(SchedulerNode,SchedulerNode)
op0_op1.writes = [MemoryDep('buf0', c0, {c0: 1024}), MemoryDep('buf1', c0, {c0: 1024})]
op0_op1.unmet_dependencies = []
op0_op1.met_dependencies = [MemoryDep('arg0_1', c0, {c0: 65536}), MemoryDep('arg1_1', c0, {c0: 524288})]
op0_op1.min_input_distance = 0
op0_op1.max_input_distance = 0
op0_op1.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[1024], stride=[1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
    buf1: ComputedBuffer
    buf1.layout = FixedLayout('cuda:0', torch.float32, size=[1024], stride=[1])
    buf1.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0_op1.snodes[0] =
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', c0, {c0: 1024})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', c0, {c0: 65536})]
op0.min_input_distance = 0
op0.max_input_distance = 

[DEBUG]:Output code: 
# AOT ID: ['7_inference']
import torch
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._C._dynamo.guards import copy_if_misaligned
from torch._C import _cuda_getCurrentRawStream as get_raw_stream
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

assert_size_stride_grouped = torch._C._dynamo.guards.assert_size_stride_grouped
empty_strided_cuda = torch._C._dynamo.guards._empty_strided_cuda
async_compile = AsyncCompile()


# kernel path: /tmp/torchinductor_root/tmpasn0oo4_/4e/c4ejtrnu5j6rkjvq54mpxb4ce4oqizdztdzisx7gkei43xwr7itr.py
# Unsorted Source Nodes: [], Original ATen: []
# Source node to ATen node mapping:
import os
from torch._inductor.runtime.runtime_utils import cache_dir

import triton
import triton.language as tl

from torch._inductor.runtime import triton_helpers, triton_heuristics
from torch._inductor.runtime.triton_helpers import libdevice, math as tl_math
from torch._inductor.run

[DEBUG]:Output code written to: /tmp/torchinductor_root/tmpasn0oo4_/4q/c4qwfybwdnxn3mk64kpnumgubsz2bnwo7uwni4cytcl4py6xwjaf.py


[INFO]:Output code written to: /tmp/torchinductor_root/tmpasn0oo4_/4q/c4qwfybwdnxn3mk64kpnumgubsz2bnwo7uwni4cytcl4py6xwjaf.py


## What changes in the scheduler

Both configurations start with the same pre-fusion IR:

```text
op0: SchedulerNode  # a.sum(-1)
  op0.group.iteration = (1024, 64)
  reads arg0_1, writes buf0 (1024)

op1: SchedulerNode  # b.sum(-1)
  op1.group.iteration = (1024, 512)
  reads arg1_1, writes buf1 (1024)
```

With combo kernels disabled, post-fusion still has two scheduler nodes, so codegen emits two Triton launches:

```python
triton_per_fused_sum_0.run(arg0_1, buf0, 1024, 64, stream=stream0)
triton_per_fused_sum_1.run(arg1_1, buf1, 1024, 512, stream=stream0)
```

With combo kernels enabled, the pass wraps the independent nodes in a `ForeachKernelSchedulerNode`:

```text
op0_op1: ForeachKernelSchedulerNode(SchedulerNode, SchedulerNode)
```

This is intentionally different from `FusedSchedulerNode`. A normal fused node combines loop bodies into one shared iteration structure. A foreach node keeps each subkernel's own loop structure and uses the combined wrapper only to get one CUDA launch.


## Codegen shape

The generated Triton kernel flattens the launch grid across subkernels. Each program id chooses which original kernel body to execute:

```python
@triton.jit
def triton_per_fused_0(in_ptr0, in_ptr1, out_ptr0, out_ptr1,
                       XBLOCK_0: tl.constexpr, XBLOCK_1: tl.constexpr):
    pid = tl.program_id(0)
    x_blocks_0 = tl.cdiv(1024, XBLOCK_0)
    num_blocks_0 = x_blocks_0
    x_blocks_1 = tl.cdiv(1024, XBLOCK_1)
    num_blocks_1 = num_blocks_0 + x_blocks_1

    if pid < num_blocks_0:
        # Subkernel 0: a.sum(dim=-1)
        local_pid = pid
        R0_BLOCK_0: tl.constexpr = 64
        ...
        tl.store(out_ptr0 + xindex, tmp4, xmask)
    elif pid < num_blocks_1:
        # Subkernel 1: b.sum(dim=-1)
        local_pid = pid - num_blocks_0
        R0_BLOCK_1: tl.constexpr = 512
        ...
        tl.store(out_ptr1 + xindex, tmp9, xmask)
```

`combo_kernel_per_subkernel_blocks=True` is the important detail in this example. The two reductions have different reduction sizes, so the combo kernel should not force both subkernels to share a single `R0_BLOCK`. Instead, the generated metadata carries per-subkernel size hints:

```python
'grid_type': 'SequentialFlattenComboKernelGrid',
'combo_grid_meta': {
    'num_kernels': 2,
    'heuristic_0': 'persistent_reduction',
    'size_hints_0': {'x': 1024, 'r0_': 64},
    'heuristic_1': 'persistent_reduction',
    'size_hints_1': {'x': 1024, 'r0_': 512},
    ...
}
```


## Where the pass fits

At a high level, combo kernel generation runs after normal scheduling has produced candidate scheduler nodes. It filters out nodes that are not suitable, groups remaining nodes by device and stream, applies distance and peak-memory gates when configured, and replaces accepted groups with `ForeachKernelSchedulerNode` instances.

This is most useful for launch-bound work: optimizer-style foreach patterns, loops over many independent tensors, and small pointwise or reduction kernels where launch overhead is a large fraction of runtime. It does not improve memory traffic the way vertical fusion does; it improves dispatch overhead by turning multiple launches into one launch with an internal `pid` dispatch.


# Loop Reordering After Fusion

Loop reordering is the simplest good example of scheduler-level fusion getting smarter about iteration order.

Consider two consumers of the same input:

```python
def f(x):
    return x.sum(dim=-1), x.t().contiguous()
```

The reduction wants to walk `x` row-major with iteration `(256, 512)`. The transpose wants to produce a contiguous output, so its natural pointwise loop walks `(512, 256)`. Both read the same input, but the `MemoryDep` expressions do not match, so the scheduler normally keeps them as two kernels.

With `torch._inductor.config.loop_ordering_after_fusion=True`, the scheduler rewrites the transpose loop to use the reduction's order. The load becomes identical to the reduction load, and the transpose moves to the store address:

```text
Before:
  sum load:       512*p0 + p1
  transpose load: p0 + 512*p1

After rewriting transpose:
  sum load:       512*p0 + p1
  transpose load: 512*p0 + p1
  transpose store: p0 + 256*p1
```

The result is one fused kernel: load each element once, accumulate the row sum, and store the transposed output from the same loop.


In [8]:
import os

os.environ.setdefault("TORCH_LOGS_FORMAT", "[%(levelname)s]:%(message)s")

import torch
from torch._inductor import config as inductor_config

inductor_config.force_disable_caches = True


def set_demo_logs():
    torch._logging.set_logs(
        fusion=True,
        ir_pre_fusion=True,
        ir_post_fusion=True,
        output_code=True,
        schedule=True,
    )


def run_loop_reordering_demo(loop_reorder: bool):
    torch._dynamo.reset()
    set_demo_logs()

    @torch.compile(fullgraph=True)
    def f(x):
        return x.sum(dim=-1), x.t().contiguous()

    x = torch.randn(256, 512, device="cuda")

    with inductor_config.patch({
        "loop_ordering_after_fusion": loop_reorder,
        "triton.unique_kernel_names": True,
    }):
        outs = f(x)
        torch.cuda.synchronize()
        return outs


for loop_reorder in (False, True):
    print(f"\n=== loop_ordering_after_fusion={loop_reorder} ===")
    _ = run_loop_reordering_demo(loop_reorder)



=== loop_ordering_after_fusion=False ===


torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', c0, {c0: 256})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', c0, {c0: 131072})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[256], stride=[1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (256, 512)
op0.sizes = ([256], [512])
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[256, 512], stride=[512, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[256], stride=[1])
class op0_loop_body:
    var_ranges = {p0: 256, p1: 512}
    index0 = 512*p0 + p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg0_1', get_index)
        reduction = ops.reduction(torch.float32, torch.float32, 'sum', load)
        get_inde

[DEBUG]:===== attempting fusion (1/10): 2 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[512]', 'sum', 'origins=OrderedSet([sum_1])', 'stack_traces = {', '  File "<cell>", line 27, in f', '    return x.sum(dim=-1), x.t().contiguous()', '           ^^^^^^^^^^^^^', '', '}'])


... (32 outputs left out)

[DEBUG]:Schedule:
 [SchedulerNode(name='op0'), SchedulerNode(name='op1')]


[DEBUG]:Output code: 
# AOT ID: ['9_inference']
import torch
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._C._dynamo.guards import copy_if_misaligned
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

assert_size_stride = torch._C._dynamo.guards.assert_size_stride
empty_strided_cuda = torch._C._dynamo.guards._empty_strided_cuda
async_compile = AsyncCompile()


# kernel path: /tmp/torchinductor_root/tmptlv1a7pp/ol/colf763mrgovuf736d57z5bcym4a4j7aqazb3wfs5z7k5cpwbvzy.py
# Topologically Sorted Source Nodes: [sum_1, t, contiguous], Original ATen: [aten.sum, aten.t, aten.clone]
# Source node to ATen node mapping:
#   contiguous => clone
#   sum_1 => sum_1
#   t => permute
# Graph fragment:
#   %arg0_1 : Tensor "f32[256, 512][512, 1]cuda:0" = PlaceHolder[target=arg0_1]
#   %sum_1 : Tensor "f32[256][1]cuda:0"[num_users=1] = call_function[target=torch.ops.aten.sum.dim_IntList](args = (%arg0_1, [-1]), kwargs = {})
#   %

[DEBUG]:Output code written to: /tmp/torchinductor_root/tmptlv1a7pp/7q/c7q4tdniput2volb4uy462xeohce4n2wjorg5wgcnmeac4dy72rt.py


[INFO]:Output code written to: /tmp/torchinductor_root/tmptlv1a7pp/7q/c7q4tdniput2volb4uy462xeohce4n2wjorg5wgcnmeac4dy72rt.py


What to look for in the logs:

- OFF: two scheduler nodes and two launches, one reduction kernel and one transpose kernel.
- ON: a `FusedSchedulerNode` where the transpose body has been rewritten into the reduction's loop order.
- In output code, the fused kernel has one `tl.load` from the input feeding both the reduction accumulation and the transposed store.

This is different from loop reindexing: reordering keeps the same factorization but changes axis order. Reindexing changes the factorization itself, for example `[16, 8192]` into `[1024, 128]`.


# Mix-Order Reduction

Mix-order reduction handles a harder case: two reductions over the same tensor along different dimensions.

```python
def f(x):
    return x.sum(dim=0), x.sum(dim=1)
```

For `x: [32768, 768]`, `sum(dim=0)` wants iteration `(768, 32768)`, while `sum(dim=1)` wants `(32768, 768)`. There is no useful loop order rewrite that makes both reductions identical. Instead, Inductor creates a special `FusedMixOrderReductions` node.

The codegen strategy is:

1. Stream the big tensor once in the fast row-major direction.
2. Produce `sum(dim=1)` directly.
3. Accumulate partials for `sum(dim=0)` into a workspace.
4. Run a small final reduction over the workspace.

So the important improvement is not "one total kernel no matter what". It is "one full pass over the large activation instead of two full passes".


In [9]:
import os

os.environ.setdefault("TORCH_LOGS_FORMAT", "[%(levelname)s]:%(message)s")

import torch
from torch._inductor import config as inductor_config

inductor_config.force_disable_caches = True


def set_demo_logs():
    torch._logging.set_logs(
        fusion=True,
        ir_pre_fusion=True,
        ir_post_fusion=True,
        output_code=True,
        schedule=True,
    )


def run_mix_order_demo(mix_order: bool):
    torch._dynamo.reset()
    set_demo_logs()

    @torch.compile(fullgraph=True)
    def f(x):
        return x.sum(dim=0), x.sum(dim=1)

    x = torch.randn(32768, 768, device="cuda")

    with inductor_config.patch({
        "triton.mix_order_reduction": mix_order,
        "triton.unique_kernel_names": True,
        "split_reductions": False,
    }):
        out = f(x)
        torch.cuda.synchronize()
        return out


for mix_order in (False, True):
    print(f"\n=== triton.mix_order_reduction={mix_order} ===")
    _ = run_mix_order_demo(mix_order)



=== triton.mix_order_reduction=False ===


torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(
[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', d0, {d0: 768})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', d0 + 768*d1, {d0: 768, d1: 32768})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[768], stride=[1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (768, 32768)
op0.sizes = ([768], [32768])
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[32768, 768], stride=[768, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[768], stride=[1])
class op0_loop_body:
    var_ranges = {p0: 768, p1: 32768}
    index0 = p0 + 768*p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index

[DEBUG]:===== attempting fusion (1/10): 2 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[32768]', 'sum', 'origins=OrderedSet([sum_1])', 'stack_traces = {', '  File "<cell>", line 27, in f', '    return x.sum(dim=0), x.sum(dim=1)', '           ^^^^^^^^^^^^', '', '}'])


[DEBUG]:  SchedulerNode(name='op1'), Reduction(['[768]', 'sum', 'origins=OrderedSet([sum_2])', 'stack_traces = {', '  File "<cell>", line 27, in f', '    return x.sum(dim=0), x.sum(dim=1)', '                         ^^^^^^^^^^^^', '', '}'])


... (29 outputs left out)

[INFO]:AFTER FUSION
op1_op0: FusedMixOrderReductions(SchedulerNode,SchedulerNode)
op1_op0.writes = [MemoryDep('buf0', d0, {d0: 768}), MemoryDep('buf1', d0, {d0: 32768})]
op1_op0.unmet_dependencies = []
op1_op0.met_dependencies = 
    [   MemoryDep('arg0_1', 768*d0 + d1, {d0: 32768, d1: 768}),
        MemoryDep('arg0_1', d0 + 768*d1, {d0: 768, d1: 32768})]
op1_op0.min_input_distance = 0
op1_op0.max_input_distance = 0
op1_op0.outputs = [
    buf1: ComputedBuffer
    buf1.layout = FixedLayout('cuda:0', torch.float32, size=[32768], stride=[1])
    buf1.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[768], stride=[1])
    buf0.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op1_op0.snodes[0] =
op1: SchedulerNode(ComputedBuffer)
op1.writes = [MemoryDep('buf1', c0, {c0: 32768})]
op1.unmet_dependencies = []
op1.met_dependencies = [MemoryDep('arg0_1', c0, {c0: 25165824})]


[DEBUG]:Output code: 
# AOT ID: ['11_inference']
import torch
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._C._dynamo.guards import copy_if_misaligned
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

assert_size_stride = torch._C._dynamo.guards.assert_size_stride
empty_strided_cuda = torch._C._dynamo.guards._empty_strided_cuda
async_compile = AsyncCompile()


# kernel path: /tmp/torchinductor_root/tmpdtththka/w5/cw5qto5u7tf2tt2sycdqiaiqvfedcjyc7a3jr6ylgvd6ryc4qkqu.py
# Topologically Sorted Source Nodes: [sum_1, sum_2], Original ATen: [aten.sum]
# Source node to ATen node mapping:
#   sum_1 => sum_1
#   sum_2 => sum_2
# Graph fragment:
#   %arg0_1 : Tensor "f32[32768, 768][768, 1]cuda:0" = PlaceHolder[target=arg0_1]
#   %sum_1 : Tensor "f32[768][1]cuda:0"[num_users=1] = call_function[target=torch.ops.aten.sum.dim_IntList](args = (%arg0_1, [0]), kwargs = {})
#   %sum_2 : Tensor "f32[32768][1]cuda:0"[num_users=1

[DEBUG]:Output code written to: /tmp/torchinductor_root/tmpdtththka/jz/cjztb3n2obvwlwi6lq5wjc7echskkqz3w6am46vdgrelbng34tdh.py


[INFO]:Output code written to: /tmp/torchinductor_root/tmpdtththka/jz/cjztb3n2obvwlwi6lq5wjc7echskkqz3w6am46vdgrelbng34tdh.py


What to look for in the logs:

- OFF: two reduction kernels, each reading the full input.
- ON: `FusedMixOrderReductions`, a main kernel that streams `x` once, and a smaller workspace finalization for the slow-axis reduction.

This pattern matters for backward passes that need reductions along different axes of the same activation: layer norm / RMS norm backward, attention backward, and weight-gradient style reductions.


# Nested Reduction

Nested reduction targets dependent reductions with different sizes, such as RMS norm followed by block amax for quantization.

```python
def f(x):
    rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + 1e-6)
    x_normed = x / rms
    return x_normed.reshape(B, D // G, G).abs().amax(dim=-1)
```

There are two reductions:

- pass 1: a large per-row RMS reduction over `D`, for example `8192` elements.
- pass 2: a smaller block reduction over `G`, for example `16` elements.

The second reduction depends on the first because it needs the normalized full-resolution values. Without nested reduction, Inductor emits a variance kernel, then a second kernel that reloads `x`, reloads the variance, normalizes, and computes block amax.

With `FusedNestedReductions`, one persistent-reduction kernel does both passes. It loads the row once, computes the RMS statistic, normalizes the register tile, reshapes that tile in registers, and reduces groups of 16 before storing the block amax.


In [10]:
import os

os.environ.setdefault("TORCH_LOGS_FORMAT", "[%(levelname)s]:%(message)s")

import torch
from torch._inductor import config as inductor_config

inductor_config.force_disable_caches = True


def set_demo_logs():
    torch._logging.set_logs(
        fusion=True,
        ir_pre_fusion=True,
        ir_post_fusion=True,
        output_code=True,
        schedule=True,
    )


def run_nested_reduction_demo(nested: bool):
    torch._dynamo.reset()
    set_demo_logs()
    B, D, G = 128, 8192, 16

    @torch.compile(fullgraph=True)
    def f(x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + 1e-6)
        x_normed = x / rms
        return x_normed.reshape(B, D // G, G).abs().amax(dim=-1)

    x = torch.randn(B, D, device="cuda")

    with inductor_config.patch({
        "triton.nested_reduction": nested,
        "triton.unique_kernel_names": True,
    }):
        out = f(x)
        torch.cuda.synchronize()
        return out


for nested in (False, True):
    print(f"\n=== triton.nested_reduction={nested} ===")
    _ = run_nested_reduction_demo(nested)



=== triton.nested_reduction=False ===


torch/_dynamo/pgo.py:605: UserWarning: dynamo_pgo force disabled by torch.compiler.config.force_disable_caches
  warn_once(


[INFO]:BEFORE FUSION
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', d0, {d0: 128})]
op0.unmet_dependencies = []
op0.met_dependencies = [MemoryDep('arg0_1', 8192*d0 + d1, {d0: 128, d1: 8192})]
op0.min_input_distance = 0
op0.max_input_distance = 0
op0.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[128, 1], stride=[1, 128])
    buf0.users = [NodeUser(node=SchedulerNode(name='op1'), can_inplace=False, is_weak=False)]
]
op0.group.device = cuda:0
op0.group.iteration = (128, 8192)
op0.sizes = ([128], [8192])
arg0_1_layout = FixedLayout('cuda:0', torch.float32, size=[128, 8192], stride=[8192, 1])
buf0_layout = FixedLayout('cuda:0', torch.float32, size=[128, 1], stride=[1, 128])
class op0_loop_body:
    var_ranges = {p0: 128, p1: 8192}
    index0 = 8192*p0 + p1
    index1 = p0
    def body(self, ops):
        get_index = self.get_index('index0')
        load = ops.load('arg0_1', get_index)
        get_index_1 = self.get_inde

[DEBUG]:===== attempting fusion (1/10): 2 nodes =====


[DEBUG]:fuse_nodes_once, candidates:


[DEBUG]:  SchedulerNode(name='op0'), Reduction(['[8192]', 'sum', 'origins=OrderedSet([mean, mul])', 'stack_traces = {', '  File "<cell>", line 28, in f', '    rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + 1e-6)', '                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^', '', '}', 'stack_traces = {', '  File "<cell>", line 28, in f', '    rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + 1e-6)', '                                ~~^~~', '', '}'])


... (29 outputs left out)

[INFO]:AFTER FUSION
op0_op1: FusedNestedReductions(SchedulerNode,SchedulerNode)
op0_op1.writes = 
    [   MemoryDep('buf0', d0, {d0: 128}),
        MemoryDep('buf1', 512*d0 + d1, {d0: 128, d1: 512})]
op0_op1.unmet_dependencies = []
op0_op1.met_dependencies = 
    [   MemoryDep('arg0_1', 8192*d0 + 16*d1 + d2, {d0: 128, d1: 512, d2: 16}),
        MemoryDep('arg0_1', 8192*d0 + d1, {d0: 128, d1: 8192})]
op0_op1.min_input_distance = 0
op0_op1.max_input_distance = 1
op0_op1.outputs = [
    buf0: ComputedBuffer
    buf0.layout = FixedLayout('cuda:0', torch.float32, size=[128, 1], stride=[1, 128])
    buf0.users = [NodeUser(node=SchedulerNode(name='op1'), can_inplace=False, is_weak=False)]
    buf1: ComputedBuffer
    buf1.layout = FixedLayout('cuda:0', torch.float32, size=[128, 512], stride=[512, 1])
    buf1.users = [NodeUser(node=OUTPUT, can_inplace=False, is_weak=False)]
]
op0_op1.snodes[0] =
op0: SchedulerNode(ComputedBuffer)
op0.writes = [MemoryDep('buf0', d0, {d0: 128})]
op0.unmet_depen

[DEBUG]:Output code: 
# AOT ID: ['13_inference']
import torch
from torch import device, empty_strided
from torch._inductor.async_compile import AsyncCompile
from torch._C._dynamo.guards import copy_if_misaligned
from torch._C import _cuda_getCurrentRawStream as get_raw_stream

assert_size_stride = torch._C._dynamo.guards.assert_size_stride
empty_strided_cuda = torch._C._dynamo.guards._empty_strided_cuda
async_compile = AsyncCompile()


# kernel path: /tmp/torchinductor_root/tmpog6o46x7/bd/cbdvambo3gtc3gs4lkqu44rt7afrvesz3u3jw3r5me5m4gwpnzi5.py
# Topologically Sorted Source Nodes: [mul, mean], Original ATen: [aten.mul, aten.mean]
# Source node to ATen node mapping:
#   mean => mean
#   mul => mul
# Graph fragment:
#   %arg0_1 : Tensor "f32[128, 8192][8192, 1]cuda:0" = PlaceHolder[target=arg0_1]
#   %mul : Tensor "f32[128, 8192][8192, 1]cuda:0"[num_users=1] = call_function[target=torch.ops.aten.mul.Tensor](args = (%arg0_1, %arg0_1), kwargs = {})
#   %mean : Tensor "f32[128, 1][1, 1]cuda:

[DEBUG]:Output code written to: /tmp/torchinductor_root/tmpog6o46x7/ah/cahb7wan3c7ekpgn2hmoxcaaisqdbhkpibojdqddnmhwba5f2se7.py


[INFO]:Output code written to: /tmp/torchinductor_root/tmpog6o46x7/ah/cahb7wan3c7ekpgn2hmoxcaaisqdbhkpibojdqddnmhwba5f2se7.py


What to look for in the logs:

- OFF: two kernels, with the full input loaded twice and the variance materialized.
- ON: `FusedNestedReductions`, with the RMS statistic and the block amax in one generated kernel.
- In output code, look for an in-register reshape of the large reduction tile followed by a smaller reduction over the grouped dimension.

This is the quantization pipeline story: RMS norm, block amax, scale calculation, and sometimes an inline-asm pack can become one kernel instead of a sequence of memory-bound kernels.



## Misc Scheduler Passes I didn't get into

- [Reorder for Peak Memory](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L2095) - on by default
- [Cudagraph Partioning](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L2107) (soon to be enabled)

- [Reorder Compute And Comm](https://github.com/pytorch/pytorch/blob/f1d636f85b1d00e73c8e6d14c10d739efdaec124/torch/_inductor/scheduler.py#L2102-L2103) (off by default, though being worked on)


## Debug Utility I did not get into

`INDUCTOR_WRITE_SCHEDULER_GRAPH=1 TORCH_COMPILE_DEBUG=1`

For the above, .sum() + .sum() example, we get the following graph. 

<center><img src="https://drisspg.github.io/nuggets/media/inductor-walkthrough/sum-sum-scheduler-graph.svg" alt="Scheduler graph for the sum + sum example: op0 computes one sum, and op1_op2 fuses the other sum with the add."></center>


